### Creation of pickles for 
- MinMaxScaler() - "scaler"
- PCA - "pca'
- kmeans model 'kmeans' 

In [1]:
#import relevant packages
import os
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from sklearn.preprocessing import MinMaxScaler



# Model Training &  saving pickles
(training model on 3 sec csv file, stored in raw data)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

df = pd.read_csv("../raw_data/Data/features_3_sec.csv") 

# Step 1: Features
X_features = df.drop(["label", "filename", "length"], axis = 1)

# Step 2: Feature Scaling
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_features), columns = X_features.columns)

# Step 3: Applying PCA
pca = PCA(n_components=24) 
X_pca = pca.fit_transform(X_scaled)

# Step 4: Applying K-Means with optimal K, saving it as 'model'
optimal_k = 6
kmeans_clustering = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans = kmeans_clustering.fit(X_pca)

### Saving scaler and models (pca and kmeans)



In [3]:
#save scaler
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

# Save models
with open('pca.pkl', 'wb') as file:
    pickle.dump(pca, file)
    
with open('kmeans.pkl', 'wb') as file:
    pickle.dump(kmeans, file)
    
print('✅Models and scaler saved successfully!')


✅Models and scaler saved successfully!


# Testing on new data 


### Feature Extraction for new data 

In [4]:
# Define file path containing audio file
file_path = '../raw_data/Data/test_mp3/mp3_no-copyright-music-happy-306601.mp3'


In [5]:
#function for extracting data (just for reference)
def extract_features(file_path):
    """
    Using librosa to load the audio file received from api, will transform data into key features, stored in a dataframe for preprocessing
    """

    #when in the .py we will need to use os.join__file__ and then set the path

    # Find audio file
    # general_path = '../../raw_data/Data'
    # file_path = f'{general_path}/genres_original/jazz/jazz.00055.wav'

    #Load and trim audio file
    y, sr = librosa.load(str(file_path))
    audio_file, _ = librosa.effects.trim(y)

    #Extract features. When relevant, calculate mean and variance
    # Length (in samples)
    length = audio_file.shape[0]

    # Chroma Frequencies
    hop_length = 5000  # Adjust for granularity
    chromagram = librosa.feature.chroma_stft(y=audio_file, sr=sr, hop_length=hop_length)
    chroma_stft_mean = np.mean(chromagram)
    chroma_stft_var = np.var(chromagram)

    # RMS Energy
    rms_values = librosa.feature.rms(y=audio_file)
    rms_mean = np.mean(rms_values)
    rms_var = np.var(rms_values)

    # Spectral Centroid
    spectral_centroids = librosa.feature.spectral_centroid(y=audio_file, sr=sr)[0]
    spectral_centroid_mean = np.mean(spectral_centroids)
    spectral_centroid_var = np.var(spectral_centroids)

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=audio_file, sr=sr)
    spectral_bandwidth_mean = np.mean(bandwidth)
    spectral_bandwidth_var = np.var(bandwidth)

    # Spectral Rolloff
    spectral_rolloff = librosa.feature.spectral_rolloff(y=audio_file, sr=sr)[0]
    rolloff_mean = np.mean(spectral_rolloff)
    rolloff_var = np.var(spectral_rolloff)

    # Zero Crossing Rate
    zero_crossings = librosa.zero_crossings(audio_file, pad=False)
    zero_crossing_rate_mean = np.mean(zero_crossings)
    zero_crossing_rate_var = np.var(zero_crossings)

    # Harmonics & Percussive Components (HPSS)
    y_harm, y_perc = librosa.effects.hpss(audio_file)
    harmony_mean = np.mean(y_harm)
    harmony_var = np.var(y_harm)
    perceptr_mean = np.mean(y_perc)
    perceptr_var = np.var(y_perc)

    # Tempo
    tempo_value, _ = librosa.beat.beat_track(y=audio_file, sr=sr)
    tempo = tempo_value.item()

    # MFCCs (20 coefficients)
    mfccs = librosa.feature.mfcc(y=audio_file, sr=sr)
    mfcc_means = np.mean(mfccs, axis=1)
    mfcc_vars = np.var(mfccs, axis=1)

    # Build feature dictionary
    #when in the .py we will ned to use os.join__file__ and then set the path
    features = {
        'filename': os.path.basename(file_path),
        'length': length,
        'chroma_stft_mean': chroma_stft_mean,
        'chroma_stft_var': chroma_stft_var,
        'rms_mean': rms_mean,
        'rms_var': rms_var,
        'spectral_centroid_mean': spectral_centroid_mean,
        'spectral_centroid_var': spectral_centroid_var,
        'spectral_bandwidth_mean': spectral_bandwidth_mean,
        'spectral_bandwidth_var': spectral_bandwidth_var,
        'rolloff_mean': rolloff_mean,
        'rolloff_var': rolloff_var,
        'zero_crossing_rate_mean': zero_crossing_rate_mean,
        'zero_crossing_rate_var': zero_crossing_rate_var,
        'harmony_mean': harmony_mean,
        'harmony_var': harmony_var,
        'perceptr_mean': perceptr_mean,
        'perceptr_var': perceptr_var,
        'tempo': tempo,
    }

    # Add MFCC features (20 coefficients)
    for i in range(len(mfcc_means)):
        features[f'mfcc{i+1}_mean'] = mfcc_means[i]
        features[f'mfcc{i+1}_var'] = mfcc_vars[i]

    # Add the label column with a default value 'no_label'
    features['label'] = 'no_label'

    print("✅ data transformed into features")

    return features

In [6]:
#implement librosa feature extraction on new data
new_X_features = pd.DataFrame([extract_features(file_path)])
new_X_features


✅ data transformed into features


,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,mp3_no-copyright-music-happy-306601.mp3,3518976,0.332648,0.097487,0.143359,0.006432,1270.587699,578479.892715,1786.362683,469111.450187,...,95.085548,1.528506,83.621414,7.481858,104.85186,1.367451,126.393951,0.426678,158.374023,no_label


### Preprocessing 

In [7]:
# Step 1: Drop columns 
new_X_features = new_X_features.drop(["label", "filename", "length"], axis = 1)

# Step 2: Load and apply ('transform') scaler 
my_scaler=pickle.load(open("scaler.pkl", "rb"))
X_new_scaled = my_scaler.transform(new_X_features)

# Step 3: Load and apply ('transform')  PCA
pca = pickle.load(open('pca.pkl', 'rb'))
X_new_pca = pca.transform(X_new_scaled)

/home/tford/.pyenv/versions/3.10.6/envs/k_means_klang/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


### Load model and predict 

In [8]:
# Load and apply ('predict') model 
kmeans = pickle.load(open('kmeans.pkl', 'rb'))
kmeans.predict(X_new_pca)

array([2], dtype=int32)

## Hooray!  We have a model!  🎉